## לקרוא את ההיסטוריה: ‎`git log`‎, ‎`git diff`‎, ‎`git status`‎

בסעיף הקודם למדנו **לכתוב** היסטוריה. עכשיו נלמד **לקרוא** אותה — כישור שנדרש הרבה יותר בפועל, כי לרוב אתם לא זוכרים בעצמכם מה קרה בכל קומיט, או שאתם קוראים היסטוריה שכתב מישהו אחר (או "אתם" של לפני חודש).

- **‎`git status`‎** — מה השתנה מאז הקומיט האחרון, ומה עדיין לא נשמר.
- **‎`git log --oneline`‎** — רשימת כל הקומיטים, כל אחד עם hash קצר (מזהה) והודעה.
- **‎`git diff`‎** (בלי ארגומנטים) — אילו שורות השתנו בתיקיית העבודה ועוד לא עברו ‎`git add`‎.
- **‎`git diff <hash1> <hash2> -- <קובץ>`‎** — בדיוק אילו שורות השתנו בין שני קומיטים, שורה-שורה.
- **‎`git show <hash>:<קובץ>`‎** — התוכן המלא של קובץ כפי שהיה באותו קומיט.

נשתמש בהם יחד כדי לפתור תעלומה: קוד שרץ בלי שגיאה, אבל מחזיר תוצאה פיזיקלית שגויה — בדיוק כמו מאגר הבאגים מסעיף 7.9, רק שהפעם הבאג יושב איפשהו **בהיסטוריה**, לא בשורה שמונחת לפניכם.

### התרחיש: ריפו שקיבלתם, עם באג מוסתר

דמיינו שקיבלתם מעמית למעבדה ריפו עם קוד לחישוב תקופת מטוטלת פשוטה, $T = 2\pi\sqrt{L/g}$, ושלושה קומיטים בהיסטוריה. הקוד רץ בלי שגיאה — אבל כדאי לבדוק את התוצאה לפני שסומכים עליה.

In [ ]:
import tempfile, os

workdir = tempfile.mkdtemp(prefix="git_mystery_")
os.chdir(workdir)

!git init -q -b main
!git config user.email "colleague@example.com"
!git config user.name "Colleague"
!git config color.ui false

In [ ]:
%%writefile pendulum.py
import numpy as np
from scipy import constants

L = 1.0  # מטר
T = 2 * np.pi * np.sqrt(L / constants.g)
print(f"T = {T:.4f} s")

In [ ]:
!git add pendulum.py
!git commit -q -m "חישוב תקופת מטוטלת פשוטה"

In [ ]:
%%writefile pendulum.py
import numpy as np
from scipy import constants

L = 1.0  # מטר
T = 2 * np.pi * np.sqrt(L) / constants.g
print(f"T = {T:.4f} s")

In [ ]:
!git add pendulum.py
!git commit -q -m "ניקוי קוד: פישוט ביטוי החישוב"

In [ ]:
%%writefile pendulum.py
'''חישוב תקופת מטוטלת פשוטה.'''
import numpy as np
from scipy import constants

L = 1.0  # מטר
T = 2 * np.pi * np.sqrt(L) / constants.g
print(f"T = {T:.4f} s")

In [ ]:
!git add pendulum.py
!git commit -q -m "הוספת docstring"

שלושה קומיטים, אף אחד מהם לא נשמע חשוד — "ניקוי קוד" ו"הוספת docstring" נשמעים כמו שינויים תמימים לגמרי. נריץ את הקובץ כפי שהוא עכשיו:

In [ ]:
!python3 pendulum.py

תקופה של $T \approx 0.64$ שניות למטוטלת באורך מטר אחד אינה סבירה: בדיקת סבירות (שלב 4 בתהליך שלמדנו בסעיף 7.9) מראה שלמטוטלת באורך מטר צפויה תקופה של כשתי שניות. נמצא בעזרת ההיסטוריה **מתי** נכנסה השגיאה.

In [ ]:
!git log --oneline

נלכוד את רשימת ה-hashes ישירות ל-Python (עם תחביר הלכידה של IPython, `!פקודה`), כדי שנוכל להשתמש בהם ב-`git diff`/`git show` בלי להעתיק-להדביק ידנית. ‎`git log --format=%H`‎ מדפיס רק את ה-hash המלא של כל קומיט, מהחדש לישן, ו-‎`reversed`‎ הופך את הסדר, כך שבאינדקס 0 נמצא הקומיט הראשון:

In [ ]:
hashes = !git log --format=%H
oldest_first = list(reversed(hashes))
for i, h in enumerate(oldest_first):
    print(i, h)

In [ ]:
# תוכן הקובץ בקומיט הראשון (לפני כל שינוי)
!git show {oldest_first[0]}:pendulum.py

In [ ]:
# מה השתנה בין הקומיט הראשון לשני?
!git diff {oldest_first[0]} {oldest_first[1]} -- pendulum.py

זהו הבאג: ‎`np.sqrt(L) / constants.g`‎ מחשב $\sqrt{L}/g$, לא $\sqrt{L/g}$ — בדיוק הבאג הראשון ממאגר הבאגים של סעיף 7.9, הפעם מוסתר בקומיט עם הודעה מטעה ("ניקוי קוד"). ‎`git diff`‎ בין הקומיט השני לשלישי (בדקו בעצמכם) יראה שהוא לא נגע בשורת החישוב כלל — כל התיקון שצריך הוא לחזור לביטוי של הקומיט הראשון.

```{image} ../../figures/13_2_log_diff.svg
:alt: The three commits in git log order with their oldest_first index, and git diff showing the line that changed
:align: center
:class: only-light
```

```{image} ../../figures/13_2_log_diff-dark.svg
:alt: The three commits in git log order with their oldest_first index, and git diff showing the line that changed
:align: center
:class: only-dark
```

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "אחרי git status שמראה שינויים לא-commit-ים, מה ההבדל בין git diff (בלי ארגומנטים) לבין git diff <hash1> <hash2>?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "אין הבדל, שתי הפקודות זהות", "correct": False, "feedback": "לא — הן משוות דברים שונים לגמרי."},
            {"answer": "git diff לבד מציג שינויים בתיקיית העבודה שעוד לא עברו git add; git diff עם שני hash-ים משווה בין שני קומיטים בהיסטוריה", "correct": True, "feedback": "נכון. כשעוד לא הרצתם git add, זו השוואה מול הקומיט האחרון."},
            {"answer": "git diff לבד עובד רק על קבצי טקסט, וגם עם hash-ים עובד על כל קובץ", "correct": False, "feedback": "לא קשור — שתי הצורות עובדות על אותם סוגי קבצים."},
            {"answer": "git diff עם hash-ים דורש חיבור לאינטרנט, בלעדיהם לא", "correct": False, "feedback": "כל הפקודות פועלות מקומית בלבד, ללא צורך באינטרנט."}
        ]
    },
    {
        "question": "מדוע חשוב לבדוק את סבירות התוצאה הפיזיקלית (כמו T=0.64s למטוטלת של מטר) כדי לגלות באג כזה?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "כי הקוד זרק שגיאה ברורה שהצביעה בדיוק על השורה הבעייתית", "correct": False, "feedback": "בדיוק ההפך — הקוד רץ בלי שום שגיאה, זה מה שהופך אותו למסוכן."},
            {"answer": "כי אין שום דרך אחרת לגלות באגים מסוג הזה", "correct": False, "feedback": "יש דרכים נוספות (כמו בדיקות אוטומטיות), אבל בדיקת סבירות פיזיקלית היא הכי נגישה וזמינה תמיד."},
            {"answer": "כי באגים של קדימות פעולות וסדר חישוב לא זורקים שגיאה — רק תוצאה שגויה בשקט, ורק היכרות עם סדר הגודל הצפוי חושפת אותם", "correct": True, "feedback": "נכון — זו בדיוק הנקודה שגם 7.9 הדגישה."},
            {"answer": "היא לא חשובה, זו רק תוספת נחמדה", "correct": False, "feedback": "היא קריטית — בלעדיה, כלל לא הייתם יודעים שיש בעיה."}
        ]
    }
]
display_quiz(questions)

### נסו בעצמכם

התאים הבאים יוצרים repo חדש עם קובץ קצר בן שלוש שורות, ‎`free_fall.py`‎, ושומרים אותו ב-commit. אחר כך הם משנים בקובץ שורה אחת, ‎`t = 2.0`‎ הופכת ל-‎`t = 3.0`‎, ומריצים ‎`git diff`‎ בלי ארגומנטים. השינוי עוד לא עבר ‎`git add`‎, ולכן הפקודה משווה את הקובץ בתיקיית העבודה ל-commit האחרון.

**לפני שאתם מריצים את התא האחרון**, כתבו אילו שורות בפלט של ‎`git diff`‎ יתחילו ב-‎`-`‎, אילו יתחילו ב-‎`+`‎, ומה יופיע בתחילת שתי השורות שלא השתנו.

In [ ]:
workdir2 = tempfile.mkdtemp(prefix="git_diff_")
os.chdir(workdir2)

!git init -q -b main
!git config user.email "student@example.com"
!git config user.name "Student"
!git config color.ui false

In [ ]:
%%writefile free_fall.py
g = 9.8     # m/s^2
t = 2.0     # s
v = g * t   # m/s

In [ ]:
!git add free_fall.py
!git commit -q -m "מהירות בנפילה חופשית"

In [ ]:
%%writefile free_fall.py
g = 9.8     # m/s^2
t = 3.0     # s
v = g * t   # m/s

In [ ]:
!git diff

`````{admonition} פתרון
:class: dropdown, tip
```text
diff --git a/free_fall.py b/free_fall.py
index 395c06a..d7ab40b 100644
--- a/free_fall.py
+++ b/free_fall.py
@@ -1,3 +1,3 @@
 g = 9.8     # m/s^2
-t = 2.0     # s
+t = 3.0     # s
 v = g * t   # m/s
```
- השורה שמתחילה ב-‎`-`‎ היא הגרסה ב-commit האחרון, ‎`t = 2.0`‎, והשורה שמתחילה ב-‎`+`‎ היא הגרסה החדשה בתיקיית העבודה, ‎`t = 3.0`‎. שינוי של שורה אחת מופיע כשתי שורות: הישנה נמחקת והחדשה נוספת.
- שתי השורות האחרות מתחילות ברווח: הן לא השתנו, ו-Git מציג אותן כהקשר. השורה ‎`v = g * t`‎ לא השתנתה כטקסט, ולכן אינה מסומנת, אף שהערך של ‎`v`‎ בהרצה משתנה.
- ארבע השורות הראשונות והשורה ‎`@@ -1,3 +1,3 @@`‎ אינן שורות של הקובץ: הן מציינות אילו קבצים מושווים והיכן בקובץ נמצא השינוי.
`````